# Pathway 1: Single Solid Analysis

This interactive tutorial walks through the simplest analysis pathway: a single rectangular waveguide solved with the Frequency Domain Solver (FDS) and accelerated with Model Order Reduction (ROM).

```
Single Solid Model  →  Frequency Domain Solver  →  Reduced Order Model
```

## 1. Create Geometry

We create a rectangular waveguide with:
- Width $a = 100$ mm
- Length $L = 200$ mm
- Mesh element size $h_{\max} = 40$ mm

In [ ]:
from geometry.primitives import RectangularWaveguide

a = 100e-3   # Width: 100 mm
L = 200e-3   # Length: 200 mm

rwg = RectangularWaveguide(a=a, L=L, maxh=0.04)
print(f"Dimensions: a={a*1e3:.0f} mm, b={rwg.b*1e3:.0f} mm, L={L*1e3:.0f} mm")
print(f"Cutoff frequency (TE10): {rwg.cutoff_frequency_TE10 / 1e9:.3f} GHz")
print(f"Mesh vertices: {rwg.mesh.nv}")

## 2. Visualise the Mesh

In [ ]:
rwg.show('mesh')

## 3. Assemble FEM Matrices

The solver:
1. Builds the $H(\text{curl})$ finite element space on the mesh
2. Solves the 2D port eigenvalue problem on each port face
3. Assembles the global stiffness matrix $\mathbf{K}$, mass matrix $\mathbf{M}$, and port excitation matrix $\mathbf{B}$

In [ ]:
from solvers.frequency_domain import FrequencyDomainSolver

fds = FrequencyDomainSolver(rwg, order=3)
fds.assemble_matrices(nmodes=1)
fds.print_info()

## 4. Solve Full-Order Model (FOM)

At each frequency $\omega_i$, the solver computes:

$$
(\mathbf{K} - \omega_i^2 \mathbf{M}) \mathbf{x}_i = \mathbf{B} \mathbf{a}_i
$$

We solve at 30 frequency points and store the snapshots for later reduction.

In [ ]:
results = fds.solve(fmin=1.5, fmax=3.0, nsamples=30, store_snapshots=True)
print(f"Solved at {len(fds.frequencies)} frequency points")
print(f"Ports: {fds.ports}")

## 5. Build Reduced-Order Model (ROM)

POD (Proper Orthogonal Decomposition) computes the SVD of the snapshot matrix and truncates to a low-rank basis:

$$
[\mathbf{x}_1, \dots, \mathbf{x}_N] = \mathbf{U} \boldsymbol{\Sigma} \mathbf{W}^H
$$

In [ ]:
from rom.reduction import ModelOrderReduction

rom = ModelOrderReduction(fds)
rom.reduce(tol=1e-6)
rom.print_info()

## 6. Solve ROM (Wideband)

The reduced system (typically rank 10–30) is solved at 500 frequency points in milliseconds.

In [ ]:
import time

t0 = time.time()
results_rom = rom.solve(fmin=1.5, fmax=3.0, nsamples=500)
print(f"ROM solve time: {(time.time() - t0)*1e3:.1f} ms for 500 frequency points")

## 7. Compare with Analytical Solution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from analytical.rectangular_waveguide import RWGAnalytical

analytical = RWGAnalytical(a=a, L=L)
frequencies = np.linspace(1.5, 3.0, 200) * 1e9
Z_ana = analytical.z_parameters(frequencies)
S_ana = analytical.s_parameters(frequencies)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

freq_ghz_fom = fds.frequencies / 1e9
freq_ghz_rom = rom.frequencies / 1e9
freq_ghz_ana = frequencies / 1e9

# S11
ax = axes[0, 0]
ax.plot(freq_ghz_ana, 20 * np.log10(np.abs(S_ana['S11']) + 1e-15),
        '-', label='Analytical', linewidth=2)
ax.plot(freq_ghz_fom, fds.get_s_db(1, 1), 'o', label='FOM', markersize=4)
ax.plot(freq_ghz_rom, rom.get_s_db(1, 1), '--', label='ROM', linewidth=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|S11| (dB)')
ax.set_title('S11 Magnitude'); ax.legend(); ax.grid(True, alpha=0.3)

# S21
ax = axes[0, 1]
ax.plot(freq_ghz_ana, 20 * np.log10(np.abs(S_ana['S21']) + 1e-15),
        '-', label='Analytical', linewidth=2)
ax.plot(freq_ghz_fom, fds.get_s_db(1, 2), 'o', label='FOM', markersize=4)
ax.plot(freq_ghz_rom, rom.get_s_db(1, 2), '--', label='ROM', linewidth=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|S21| (dB)')
ax.set_title('S21 Magnitude'); ax.legend(); ax.grid(True, alpha=0.3)

# Z11
ax = axes[1, 0]
ax.plot(freq_ghz_ana, 20 * np.log10(np.abs(Z_ana['Z11']) + 1e-15),
        '-', label='Analytical', linewidth=2)
ax.plot(freq_ghz_fom, fds.get_z_db(1, 1), 'o', label='FOM', markersize=4)
ax.plot(freq_ghz_rom, rom.get_z_db(1, 1), '--', label='ROM', linewidth=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|Z11| (dB)')
ax.set_title('Z11 Magnitude'); ax.legend(); ax.grid(True, alpha=0.3)

# Z21
ax = axes[1, 1]
ax.plot(freq_ghz_ana, 20 * np.log10(np.abs(Z_ana['Z21']) + 1e-15),
        '-', label='Analytical', linewidth=2)
ax.plot(freq_ghz_fom, fds.get_z_db(1, 2), 'o', label='FOM', markersize=4)
ax.plot(freq_ghz_rom, rom.get_z_db(1, 2), '--', label='ROM', linewidth=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|Z21| (dB)')
ax.set_title('Z21 Magnitude'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('FOM vs ROM vs Analytical', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Error Analysis

In [ ]:
# ROM vs FOM error
errors = rom.compute_error(fds)
print("ROM vs FOM errors:")
for key, err in errors.items():
    print(f"  {key}: {err:.4e}")

# FOM vs Analytical (Z11)
Z_ana_interp = analytical.z_parameters(fds.frequencies)
Z11_fom = fds.get_param('Z', '1(1)1(1)')
Z11_ana = Z_ana_interp['Z11']
rel_error = np.abs(Z11_fom - Z11_ana) / (np.abs(Z11_ana) + 1e-15)
print(f"\nFOM vs Analytical (Z11):")
print(f"  Mean relative error: {np.mean(rel_error)*100:.2f}%")
print(f"  Max relative error:  {np.max(rel_error)*100:.2f}%")

## 9. Eigenfrequencies

In [ ]:
freqs_fom = fds.get_resonant_frequencies() / 1e9
freqs_rom = rom.get_resonant_frequencies() / 1e9

print("First 5 resonant modes:")
print(f"{'Mode':>6} {'FOM (GHz)':>12} {'ROM (GHz)':>12}")
for i, (f_fom, f_rom) in enumerate(zip(sorted(freqs_fom)[:5], sorted(freqs_rom)[:5])):
    print(f"{i+1:>6} {f_fom:>12.4f} {f_rom:>12.4f}")

## Summary

In this tutorial we:
1. Created a rectangular waveguide geometry
2. Assembled K, M, B matrices using the Frequency Domain Solver
3. Solved the FOM at 30 frequency points
4. Reduced the system via POD to a compact ROM
5. Solved the ROM at 500 points in milliseconds
6. Validated against the analytical solution

**Next:** Try [Pathway 3: FOM Concatenation](pathway3_fom_concatenation.ipynb) for multi-domain analysis.